1. README OF DASHBOARD


 Tools and Technologies Used

Programming Language

Python

Frontend / UI

Gradio:  used to build the interactive dashboard interface

Backend

FastAPI:  used to create APIs and handle server logic

Database

SQLite: used to store employee data locally

Deployment

NgroK:  used to generate a public link for the dashboard

Data Processing

Pandas :  used for table handling and Excel export

Excel Export

OpenPyXL : used to create Excel files



2: These are the main Python libraries used in your dashboard:

gradio
fastapi
uvicorn
pandas
sqlite3
openpyxl
pyngrok
nest_asyncio
datetime
random
threading
os
time



3: These commands are used to set up the dashboard environment.

Core Installation
pip install gradio pandas openpyxl
Backend Installation
pip install fastapi uvicorn pyngrok nest-asyncio


4. Database — How Data is Stored

Database Type

SQLite

Database File

hr_database.db

Table Name

employees

Table Fields

id — Primary Key
name — Employee Name
department — Department
task — Assigned Task
status — Task Status



Data is stored locally in the database
Data remains saved after refresh
No external database server required
Lightweight and easy to manage

5: How Deployment on Ngrok is Done

Ngrok is used to make the dashboard accessible online.

Steps
Import Ngrok
from pyngrok import ngrok
Stop old sessions
ngrok.kill()
Start tunnel
ngrok.connect(7860)


6: How Dashboard Connects to Database

Flow of data:

User Action
→ Gradio Interface
→ FastAPI Backend
→ SQLite Database
→ Updated Dashboard


7: What Happens When Dashboard is Refreshed

When the dashboard refreshes:

Database is reloaded
Updated employee list is displayed
Deleted employees disappear
New employees appear
Table stays synchronized with database
. System Architecture (Simple)

User
↓
Gradio Dashboard
↓
FastAPI Backend
↓
SQLite Database
↓
Excel Export



Developed an HR Management Dashboard using Python, Gradio, FastAPI, and SQLite with real-time employee management, Excel export functionality, and public deployment using Ngrok.

In [ ]:
!pip install gradio pandas openpyxl pyngrok --quiet

In [ ]:


!pip install gradio pyngrok openpyxl --quiet



import gradio as gr
import pandas as pd
import random
import datetime
import socket
import os
from pyngrok import ngrok



print("Cleaning old sessions...")

try:
    ngrok.kill()
except:
    pass

os.system("pkill -f gradio")
os.system("pkill -f ngrok")

def find_free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port

port = find_free_port()

print("Using PORT:", port)



ngrok.set_auth_token(
    "3BrJPwPah3b389uAFPPR6mpZANR_8653AXdenSH2ayYjv6Feb"
)



def create_initial_data():

    today = datetime.date.today()

    df = pd.DataFrame({

        "Employer":
            ["NextGen Talent Solutions Pvt Ltd"] * 5,

        "Virtual HR":
            [
                "Rohan Sharma",
                "Ishan Singh",
                "Mohan Verma",
                "Rohit Kumar",
                "Vandana Gupta"
            ],

        "Department":
            [
                "Recruitment",
                "HR Operations",
                "Verification",
                "Talent Acquisition",
                "HR Support"
            ],

        "Assigned Task":
            [
                "Screen Resume",
                "Conduct Interview",
                "Verify Documents",
                "Update Candidate Status",
                "Send Offer Letter"
            ],

        "Priority":
            ["High","Medium","Low","High","Medium"],

        "Status":
            [
                "Assigned",
                "In Progress",
                "Completed",
                "In Progress",
                "Assigned"
            ],

        "Progress (%)":
            [
                random.randint(30,100)
                for _ in range(5)
            ],

        "Assigned Date":
            [today] * 5

    })

    df["Performance Score"] = [
        random.randint(70,100)
        for _ in range(len(df))
    ]

    df["Last Updated"] = datetime.datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    return df


dashboard_df = create_initial_data()



def calculate_stats(df):

    total = len(df)

    completed = len(df[df["Status"] == "Completed"])

    in_progress = len(df[df["Status"] == "In Progress"])

    pending = total - completed - in_progress

    return total, completed, in_progress, pending


def refresh_dashboard():

    total, completed, in_progress, pending = \
        calculate_stats(dashboard_df)

    return (
        dashboard_df,
        total,
        completed,
        in_progress,
        pending
    )


def add_employee(name):

    global dashboard_df

    if name.strip() == "":
        return refresh_dashboard()

    new_row = {

        "Employer":
            "NextGen Talent Solutions Pvt Ltd",

        "Virtual HR":
            name,

        "Department":
            "Recruitment",

        "Assigned Task":
            "Screen Resume",

        "Priority":
            "High",

        "Status":
            "Assigned",

        "Progress (%)":
            random.randint(30,100),

        "Assigned Date":
            datetime.date.today(),

        "Performance Score":
            random.randint(70,100),

        "Last Updated":
            datetime.datetime.now()
            .strftime("%Y-%m-%d %H:%M:%S")
    }

    dashboard_df = pd.concat(
        [dashboard_df,
         pd.DataFrame([new_row])],
        ignore_index=True
    )

    return (
        *refresh_dashboard(),
        f"{name} added successfully"
    )


def delete_employee(name):

    global dashboard_df

    dashboard_df = dashboard_df[
        dashboard_df["Virtual HR"] != name
    ]

    return (
        *refresh_dashboard(),
        f"{name} deleted successfully"
    )


def export_to_excel():

    dashboard_df.to_excel(
        "HR_Dashboard_Report.xlsx",
        index=False
    )

    return "Excel downloaded successfully"



custom_css = """

body {
    background-color: #eef3ff;
}

.title {
    text-align:center;
    font-size:36px;
    font-weight:bold;
    color:white;
    background: linear-gradient(
        90deg,
        #007bff,
        #6f42c1
    );
    padding:16px;
    border-radius:10px;
}

"""

with gr.Blocks(css=custom_css) as demo:

    gr.Markdown(
        "<div class='title'>SMART HR MANAGEMENT DASHBOARD</div>"
    )

    refresh_btn = gr.Button("Refresh Dashboard")

    export_btn = gr.Button("Export to Excel")

    name_input = gr.Textbox(
        label="Employee Name"
    )

    add_btn = gr.Button("Add Employee")

    delete_btn = gr.Button("Delete Employee")

    total_box = gr.Number(label="Total Tasks")

    completed_box = gr.Number(label="Completed")

    progress_box = gr.Number(label="In Progress")

    pending_box = gr.Number(label="Pending")

    table = gr.Dataframe()

    message = gr.Textbox(label="System Message")

    refresh_btn.click(
        refresh_dashboard,
        outputs=[
            table,
            total_box,
            completed_box,
            progress_box,
            pending_box
        ]
    )

    add_btn.click(
        add_employee,
        inputs=name_input,
        outputs=[
            table,
            total_box,
            completed_box,
            progress_box,
            pending_box,
            message
        ]
    )

    delete_btn.click(
        delete_employee,
        inputs=name_input,
        outputs=[
            table,
            total_box,
            completed_box,
            progress_box,
            pending_box,
            message
        ]
    )

    export_btn.click(
        export_to_excel,
        outputs=message
    )



public_url = ngrok.connect(port)

print("\nPUBLIC DASHBOARD LINK:")
print(public_url)

demo.launch(
    server_port=port,
    server_name="0.0.0.0",
    share=False
)

Cleaning old sessions...
Using PORT: 51045


/tmp/ipykernel_1176/891158935.py:267: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css) as demo:



PUBLIC DASHBOARD LINK:
NgrokTunnel: "https://harriet-unrefilled-superboldly.ngrok-free.dev" -> "http://localhost:51045"
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

DEPLOYMENT LIBRARIES
Backend server ready Database support ready Deployment support ready

In [ ]:
!pip install fastapi uvicorn sqlite-utils nest-asyncio pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.5/68.5 kB 2.4 MB/s eta 0:00:00


CREATE SQLITE DATABASE FOR DASHBOARD

In [ ]:
import sqlite3

# create database
conn = sqlite3.connect("hr_database.db")

cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS employees (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    department TEXT,
    task TEXT,
    status TEXT
)
""")

conn.commit()

print("Database created successfully")

Database created successfully


INSERT SAMPLE DATA


In [ ]:
import sqlite3

conn = sqlite3.connect("hr_database.db")
cursor = conn.cursor()

employees = [
    ("Rohan", "Recruitment", "Screen Resume", "Assigned"),
    ("Ishan", "HR", "Interview", "In Progress"),
    ("Mohan", "Verification", "Verify Documents", "Completed")
]

cursor.executemany(
    "INSERT INTO employees (name, department, task, status) VALUES (?, ?, ?, ?)",
    employees
)

conn.commit()

print("Data inserted successfully")

Data inserted successfully


CREATE FAST API BACKENED

In [ ]:
from fastapi import FastAPI
import sqlite3

app = FastAPI()

@app.get("/")
def home():
    return {"message": "HR Management API Running"}

@app.get("/employees")
def get_employees():

    conn = sqlite3.connect("hr_database.db")
    cursor = conn.cursor()

    cursor.execute("SELECT * FROM employees")

    data = cursor.fetchall()

    return {
        "employees": data
    }

In [ ]:


import nest_asyncio
import uvicorn
import threading

# Fix event loop conflict
nest_asyncio.apply()

print("Starting FastAPI server on port 8000...")

# Function to run server
def run_server():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="info"
    )

# Run server in background thread
server_thread = threading.Thread(target=run_server)
server_thread.start()

print("FastAPI server started successfully!")

Starting FastAPI server on port 8000...
FastAPI server started successfully!


INFO:     Started server process [1176]
INFO:     Waiting for application startup.
